In [ ]:
import { useState, useEffect, useRef } from "react";
import { LineChart, Line, XAxis, YAxis, Tooltip, ResponsiveContainer, ReferenceLine, Area, AreaChart, CartesianGrid } from "recharts";

const STOCKS = {
  AAPL: { name: "Apple Inc.", base: 178, color: "#00d4aa" },
  TSLA: { name: "Tesla Inc.", base: 245, color: "#ff6b35" },
  GOOGL: { name: "Alphabet Inc.", base: 142, color: "#4da6ff" },
  MSFT: { name: "Microsoft Corp.", base: 415, color: "#a78bfa" },
  AMZN: { name: "Amazon.com", base: 185, color: "#fbbf24" },
  NVDA: { name: "NVIDIA Corp.", base: 875, color: "#34d399" },
};

function generateStockData(ticker, days) {
  const stock = STOCKS[ticker];
  let price = stock.base;
  const data = [];
  const now = new Date();
  for (let i = days; i >= 0; i--) {
    const date = new Date(now);
    date.setDate(date.getDate() - i);
    const change = (Math.random() - 0.48) * price * 0.025;
    price = Math.max(price + change, price * 0.5);
    const open = price * (1 + (Math.random() - 0.5) * 0.01);
    const high = Math.max(price, open) * (1 + Math.random() * 0.015);
    const low = Math.min(price, open) * (1 - Math.random() * 0.015);
    const volume = Math.floor(Math.random() * 80000000 + 20000000);
    data.push({
      date: date.toLocaleDateString("en-US", { month: "short", day: "numeric" }),
      close: parseFloat(price.toFixed(2)),
      open: parseFloat(open.toFixed(2)),
      high: parseFloat(high.toFixed(2)),
      low: parseFloat(low.toFixed(2)),
      volume,
    });
  }
  // Add SMA
  for (let i = 0; i < data.length; i++) {
    if (i >= 19) {
      const slice = data.slice(i - 19, i + 1).map(d => d.close);
      data[i].sma20 = parseFloat((slice.reduce((a, b) => a + b, 0) / 20).toFixed(2));
    }
    if (i >= 49) {
      const slice = data.slice(i - 49, i + 1).map(d => d.close);
      data[i].sma50 = parseFloat((slice.reduce((a, b) => a + b, 0) / 50).toFixed(2));
    }
  }
  return data;
}

function computeRSI(data) {
  const closes = data.map(d => d.close);
  const rsiData = closes.map((_, i) => {
    if (i < 14) return { ...data[i], rsi: null };
    const slice = closes.slice(i - 14, i + 1);
    let gains = 0, losses = 0;
    for (let j = 1; j < slice.length; j++) {
      const diff = slice[j] - slice[j - 1];
      if (diff > 0) gains += diff;
      else losses -= diff;
    }
    const rs = gains / (losses || 0.0001);
    return { ...data[i], rsi: parseFloat((100 - 100 / (1 + rs)).toFixed(1)) };
  });
  return rsiData;
}

const CustomTooltip = ({ active, payload, label, color }) => {
  if (active && payload && payload.length) {
    const d = payload[0].payload;
    return (
      <div style={{
        background: "rgba(10,12,20,0.97)",
        border: `1px solid ${color}40`,
        borderRadius: 8,
        padding: "10px 14px",
        fontSize: 12,
        color: "#e2e8f0",
        fontFamily: "'Space Mono', monospace",
        boxShadow: `0 0 20px ${color}20`
      }}>
        <div style={{ color, fontWeight: 700, marginBottom: 4 }}>{label}</div>
        <div>Close: <span style={{ color: "#fff" }}>${d.close?.toFixed(2)}</span></div>
        {d.open && <div>Open: <span style={{ color: "#94a3b8" }}>${d.open?.toFixed(2)}</span></div>}
        {d.high && <div>High: <span style={{ color: "#34d399" }}>${d.high?.toFixed(2)}</span></div>}
        {d.low && <div>Low: <span style={{ color: "#f87171" }}>${d.low?.toFixed(2)}</span></div>}
        {d.sma20 && <div>SMA20: <span style={{ color: "#fbbf24" }}>${d.sma20}</span></div>}
        {d.volume && <div>Vol: <span style={{ color: "#94a3b8" }}>{(d.volume / 1e6).toFixed(1)}M</span></div>}
      </div>
    );
  }
  return null;
};

const RSITooltip = ({ active, payload, label }) => {
  if (active && payload && payload.length && payload[0].value) {
    const rsi = payload[0].value;
    const rsiColor = rsi > 70 ? "#f87171" : rsi < 30 ? "#34d399" : "#94a3b8";
    return (
      <div style={{
        background: "rgba(10,12,20,0.97)",
        border: "1px solid #334155",
        borderRadius: 6,
        padding: "8px 12px",
        fontSize: 11,
        fontFamily: "'Space Mono', monospace"
      }}>
        <span style={{ color: "#64748b" }}>{label}: </span>
        <span style={{ color: rsiColor, fontWeight: 700 }}>RSI {rsi}</span>
      </div>
    );
  }
  return null;
};

export default function StockTrendApp() {
  const [ticker, setTicker] = useState("AAPL");
  const [range, setRange] = useState(90);
  const [data, setData] = useState([]);
  const [loading, setLoading] = useState(false);
  const [showSMA, setShowSMA] = useState(true);
  const [tab, setTab] = useState("price");

  const stock = STOCKS[ticker];
  const color = stock.color;

  useEffect(() => {
    setLoading(true);
    setTimeout(() => {
      const raw = generateStockData(ticker, range);
      setData(computeRSI(raw));
      setLoading(false);
    }, 400);
  }, [ticker, range]);

  const lastPrice = data[data.length - 1]?.close || 0;
  const firstPrice = data[0]?.close || 0;
  const change = lastPrice - firstPrice;
  const changePct = firstPrice ? ((change / firstPrice) * 100).toFixed(2) : 0;
  const isUp = change >= 0;
  const high = Math.max(...data.map(d => d.high || d.close));
  const low = Math.min(...data.map(d => d.low || d.close));
  const avgVol = data.length ? (data.reduce((s, d) => s + (d.volume || 0), 0) / data.length / 1e6).toFixed(1) : 0;
  const currentRSI = data[data.length - 1]?.rsi;
  const rsiSignal = currentRSI > 70 ? "Overbought" : currentRSI < 30 ? "Oversold" : "Neutral";
  const rsiColor = currentRSI > 70 ? "#f87171" : currentRSI < 30 ? "#34d399" : "#64748b";

  const displayData = data.filter((_, i) => i % Math.max(1, Math.floor(data.length / 80)) === 0);

  return (
    <div style={{
      minHeight: "100vh",
      background: "#060810",
      fontFamily: "'Space Mono', monospace",
      color: "#e2e8f0",
      padding: "0",
    }}>
      <style>{`
        @import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Syne:wght@700;800&display=swap');
        * { box-sizing: border-box; margin: 0; padding: 0; }
        ::-webkit-scrollbar { width: 4px; }
        ::-webkit-scrollbar-track { background: #0f1117; }
        ::-webkit-scrollbar-thumb { background: #334155; border-radius: 4px; }
        .ticker-btn {
          background: transparent;
          border: 1px solid #1e293b;
          color: #64748b;
          padding: 6px 14px;
          border-radius: 6px;
          cursor: pointer;
          font-family: 'Space Mono', monospace;
          font-size: 12px;
          font-weight: 700;
          transition: all 0.18s;
          letter-spacing: 0.5px;
        }
        .ticker-btn:hover { border-color: #334155; color: #94a3b8; }
        .ticker-btn.active { color: #060810; font-weight: 700; }
        .range-btn {
          background: transparent;
          border: 1px solid #1e293b;
          color: #475569;
          padding: 4px 12px;
          border-radius: 4px;
          cursor: pointer;
          font-family: 'Space Mono', monospace;
          font-size: 11px;
          transition: all 0.15s;
        }
        .range-btn:hover { border-color: #334155; color: #64748b; }
        .range-btn.active { border-color: #334155; color: #e2e8f0; background: #1e293b; }
        .tab-btn {
          background: transparent;
          border: none;
          color: #475569;
          padding: 8px 20px;
          cursor: pointer;
          font-family: 'Space Mono', monospace;
          font-size: 11px;
          font-weight: 700;
          letter-spacing: 1px;
          border-bottom: 2px solid transparent;
          transition: all 0.15s;
        }
        .tab-btn:hover { color: #94a3b8; }
        .tab-btn.active { color: #e2e8f0; border-bottom-color: var(--accent); }
        .stat-card {
          background: rgba(255,255,255,0.02);
          border: 1px solid #1e293b;
          border-radius: 8px;
          padding: 14px 18px;
          flex: 1;
          min-width: 100px;
        }
        @keyframes fadeIn { from { opacity: 0; transform: translateY(8px); } to { opacity: 1; transform: translateY(0); } }
        .chart-area { animation: fadeIn 0.4s ease; }
        @keyframes pulse { 0%,100% { opacity: 1; } 50% { opacity: 0.4; } }
        .loading-dot { animation: pulse 1s infinite; }
      `}</style>

      {/* Header */}
      <div style={{
        borderBottom: "1px solid #0f172a",
        padding: "16px 28px",
        display: "flex",
        alignItems: "center",
        justifyContent: "space-between",
        background: "rgba(6,8,16,0.9)",
        backdropFilter: "blur(12px)",
        position: "sticky",
        top: 0,
        zIndex: 10,
      }}>
        <div style={{ display: "flex", alignItems: "center", gap: 12 }}>
          <div style={{
            width: 32, height: 32,
            background: `linear-gradient(135deg, ${color}, ${color}88)`,
            borderRadius: 8,
            display: "flex", alignItems: "center", justifyContent: "center",
            fontSize: 14, fontWeight: 700, color: "#060810"
          }}>▲</div>
          <span style={{ fontFamily: "'Syne', sans-serif", fontWeight: 800, fontSize: 18, letterSpacing: "-0.5px" }}>
            TREND<span style={{ color }}>PULSE</span>
          </span>
        </div>
        <div style={{ fontSize: 10, color: "#334155", letterSpacing: 2 }}>STOCK ANALYTICS</div>
      </div>

      <div style={{ padding: "24px 28px", maxWidth: 1100, margin: "0 auto" }}>

        {/* Ticker selector */}
        <div style={{ display: "flex", gap: 8, flexWrap: "wrap", marginBottom: 28 }}>
          {Object.entries(STOCKS).map(([t, s]) => (
            <button
              key={t}
              className={`ticker-btn ${ticker === t ? "active" : ""}`}
              style={ticker === t ? { background: s.color, borderColor: s.color, color: "#060810" } : {}}
              onClick={() => setTicker(t)}
            >
              {t}
            </button>
          ))}
        </div>

        {/* Price header */}
        <div style={{ marginBottom: 24 }}>
          <div style={{ display: "flex", alignItems: "baseline", gap: 16, flexWrap: "wrap" }}>
            <div>
              <div style={{ fontSize: 11, color: "#475569", letterSpacing: 2, marginBottom: 4 }}>{stock.name.toUpperCase()}</div>
              <div style={{ fontFamily: "'Syne', sans-serif", fontSize: 42, fontWeight: 800, color: "#f8fafc", lineHeight: 1 }}>
                ${lastPrice.toFixed(2)}
              </div>
            </div>
            <div style={{
              display: "flex", alignItems: "center", gap: 6,
              background: isUp ? "#052e1650" : "#2d0a0a50",
              border: `1px solid ${isUp ? "#16a34a40" : "#dc262640"}`,
              borderRadius: 6, padding: "6px 12px"
            }}>
              <span style={{ fontSize: 16, color: isUp ? "#34d399" : "#f87171" }}>{isUp ? "▲" : "▼"}</span>
              <span style={{ color: isUp ? "#34d399" : "#f87171", fontSize: 14, fontWeight: 700 }}>
                {isUp ? "+" : ""}{change.toFixed(2)} ({isUp ? "+" : ""}{changePct}%)
              </span>
            </div>
          </div>
        </div>

        {/* Stats row */}
        <div style={{ display: "flex", gap: 10, marginBottom: 24, flexWrap: "wrap" }}>
          {[
            { label: "52W HIGH", value: `$${high.toFixed(2)}`, color: "#34d399" },
            { label: "52W LOW", value: `$${low.toFixed(2)}`, color: "#f87171" },
            { label: "AVG VOL", value: `${avgVol}M`, color: "#94a3b8" },
            { label: "RSI(14)", value: currentRSI ? `${currentRSI}` : "—", color: rsiColor },
            { label: "SIGNAL", value: rsiSignal, color: rsiColor },
          ].map(s => (
            <div key={s.label} className="stat-card">
              <div style={{ fontSize: 9, color: "#334155", letterSpacing: 2, marginBottom: 6 }}>{s.label}</div>
              <div style={{ fontSize: 16, fontWeight: 700, color: s.color }}>{s.value}</div>
            </div>
          ))}
        </div>

        {/* Chart container */}
        <div style={{
          background: "rgba(255,255,255,0.015)",
          border: "1px solid #0f172a",
          borderRadius: 12,
          overflow: "hidden",
        }}>
          {/* Tabs + Range */}
          <div style={{
            display: "flex", justifyContent: "space-between", alignItems: "center",
            borderBottom: "1px solid #0f172a", padding: "0 8px",
            "--accent": color,
          }}>
            <div style={{ display: "flex" }}>
              {["price", "rsi", "volume"].map(t => (
                <button
                  key={t}
                  className={`tab-btn ${tab === t ? "active" : ""}`}
                  style={tab === t ? { "--accent": color, borderBottomColor: color, color: "#e2e8f0" } : {}}
                  onClick={() => setTab(t)}
                >
                  {t.toUpperCase()}
                </button>
              ))}
            </div>
            <div style={{ display: "flex", gap: 6, padding: "8px 8px" }}>
              {[
                { label: "1M", days: 30 },
                { label: "3M", days: 90 },
                { label: "6M", days: 180 },
                { label: "1Y", days: 365 },
              ].map(r => (
                <button
                  key={r.label}
                  className={`range-btn ${range === r.days ? "active" : ""}`}
                  onClick={() => setRange(r.days)}
                >{r.label}</button>
              ))}
            </div>
          </div>

          <div style={{ padding: "20px 8px 8px", height: 320 }}>
            {loading ? (
              <div style={{ display: "flex", alignItems: "center", justifyContent: "center", height: "100%", gap: 8 }}>
                {[0, 1, 2].map(i => (
                  <div key={i} className="loading-dot" style={{
                    width: 8, height: 8, borderRadius: "50%",
                    background: color, animationDelay: `${i * 0.2}s`
                  }} />
                ))}
              </div>
            ) : (
              <div className="chart-area" style={{ height: "100%" }}>
                {tab === "price" && (
                  <ResponsiveContainer width="100%" height="100%">
                    <AreaChart data={displayData}>
                      <defs>
                        <linearGradient id="priceGrad" x1="0" y1="0" x2="0" y2="1">
                          <stop offset="5%" stopColor={color} stopOpacity={0.18} />
                          <stop offset="95%" stopColor={color} stopOpacity={0} />
                        </linearGradient>
                      </defs>
                      <CartesianGrid stroke="#0f172a" strokeDasharray="0" vertical={false} />
                      <XAxis dataKey="date" tick={{ fontSize: 10, fill: "#334155", fontFamily: "Space Mono" }} tickLine={false} axisLine={false} interval="preserveStartEnd" />
                      <YAxis tick={{ fontSize: 10, fill: "#334155", fontFamily: "Space Mono" }} tickLine={false} axisLine={false} width={60} tickFormatter={v => `$${v}`} domain={["auto", "auto"]} />
                      <Tooltip content={<CustomTooltip color={color} />} />
                      <Area type="monotone" dataKey="close" stroke={color} strokeWidth={2} fill="url(#priceGrad)" dot={false} />
                      {showSMA && <Line type="monotone" dataKey="sma20" stroke="#fbbf24" strokeWidth={1.5} dot={false} strokeDasharray="4 2" />}
                    </AreaChart>
                  </ResponsiveContainer>
                )}
                {tab === "rsi" && (
                  <ResponsiveContainer width="100%" height="100%">
                    <LineChart data={displayData.filter(d => d.rsi)}>
                      <CartesianGrid stroke="#0f172a" strokeDasharray="0" vertical={false} />
                      <XAxis dataKey="date" tick={{ fontSize: 10, fill: "#334155", fontFamily: "Space Mono" }} tickLine={false} axisLine={false} interval="preserveStartEnd" />
                      <YAxis domain={[0, 100]} tick={{ fontSize: 10, fill: "#334155", fontFamily: "Space Mono" }} tickLine={false} axisLine={false} width={36} />
                      <Tooltip content={<RSITooltip />} />
                      <ReferenceLine y={70} stroke="#f87171" strokeDasharray="4 2" strokeOpacity={0.5} label={{ value: "OB 70", fill: "#f87171", fontSize: 9, fontFamily: "Space Mono" }} />
                      <ReferenceLine y={30} stroke="#34d399" strokeDasharray="4 2" strokeOpacity={0.5} label={{ value: "OS 30", fill: "#34d399", fontSize: 9, fontFamily: "Space Mono" }} />
                      <ReferenceLine y={50} stroke="#334155" strokeDasharray="2 4" strokeOpacity={0.4} />
                      <Line type="monotone" dataKey="rsi" stroke={color} strokeWidth={2} dot={false} />
                    </LineChart>
                  </ResponsiveContainer>
                )}
                {tab === "volume" && (
                  <ResponsiveContainer width="100%" height="100%">
                    <AreaChart data={displayData}>
                      <defs>
                        <linearGradient id="volGrad" x1="0" y1="0" x2="0" y2="1">
                          <stop offset="5%" stopColor={color} stopOpacity={0.25} />
                          <stop offset="95%" stopColor={color} stopOpacity={0.02} />
                        </linearGradient>
                      </defs>
                      <CartesianGrid stroke="#0f172a" strokeDasharray="0" vertical={false} />
                      <XAxis dataKey="date" tick={{ fontSize: 10, fill: "#334155", fontFamily: "Space Mono" }} tickLine={false} axisLine={false} interval="preserveStartEnd" />
                      <YAxis tick={{ fontSize: 10, fill: "#334155", fontFamily: "Space Mono" }} tickLine={false} axisLine={false} width={60} tickFormatter={v => `${(v / 1e6).toFixed(0)}M`} />
                      <Tooltip content={({ active, payload, label }) => active && payload?.length ? (
                        <div style={{ background: "rgba(10,12,20,0.97)", border: `1px solid ${color}40`, borderRadius: 8, padding: "8px 12px", fontSize: 11, fontFamily: "Space Mono", color: "#e2e8f0" }}>
                          <div style={{ color }}>{label}</div>
                          <div>Vol: {(payload[0].value / 1e6).toFixed(2)}M</div>
                        </div>
                      ) : null} />
                      <Area type="monotone" dataKey="volume" stroke={color} strokeWidth={1.5} fill="url(#volGrad)" dot={false} />
                    </AreaChart>
                  </ResponsiveContainer>
                )}
              </div>
            )}
          </div>

          {/* Toggle SMA */}
          {tab === "price" && (
            <div style={{ padding: "8px 20px 16px", display: "flex", gap: 16, alignItems: "center" }}>
              <button
                onClick={() => setShowSMA(!showSMA)}
                style={{
                  background: showSMA ? "#1e1a0620" : "transparent",
                  border: `1px solid ${showSMA ? "#fbbf2460" : "#1e293b"}`,
                  color: showSMA ? "#fbbf24" : "#334155",
                  padding: "3px 10px", borderRadius: 4,
                  cursor: "pointer", fontSize: 10, fontFamily: "Space Mono",
                  display: "flex", alignItems: "center", gap: 6
                }}
              >
                <span style={{ display: "inline-block", width: 16, height: 2, background: "#fbbf24", opacity: showSMA ? 1 : 0.3, borderRadius: 1 }} />
                SMA 20
              </button>
              <span style={{ fontSize: 9, color: "#1e293b", letterSpacing: 1 }}>INDICATORS</span>
            </div>
          )}
        </div>

        {/* Footer note */}
        <div style={{ marginTop: 16, fontSize: 9, color: "#1e293b", letterSpacing: 1, textAlign: "center" }}>
          SIMULATED DATA FOR DEMONSTRATION PURPOSES ONLY · NOT FINANCIAL ADVICE
        </div>
      </div>
    </div>
  );
}


: 